In [12]:
!pip install ipywidgets --upgrade
!pip install pandas transformers torchxa!pip install ipywidgets --upgrade
!pip install pandas transformers torch

# Disable Hugging Face tokenizers parallelism warning
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

zsh:1: permission denied: pip
zsh:1: permission denied: pip
zsh:1: permission denied: pip


In [17]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the CSV file
csv_file = "text_summarization/customer_reviews.csv"  # Replace with your file path
data = pd.read_csv(csv_file)

# Ensure the 'reviews' column is treated as strings, in case there are non-string entries
data['reviews'] = data['reviews'].astype(str)

# Display data types to ensure everything looks correct
print(data.dtypes)

# Optionally, display the first few rows of the data to ensure it loaded correctly
print(data.head())

url        object
reviews    object
dtype: object
                                                 url  \
0  https://www.angi.com/companylist/us/tx/argyle/...   
1                                                NaN   
2  https://www.angi.com/companylist/us/md/rosedal...   
3  https://www.angi.com/companylist/us/md/bethesd...   
4  https://www.angi.com/companylist/us/or/beavert...   

                                             reviews  
0  Although most of the issues were not safety co...  
1  I am very pleased with the quality of work. My...  
2  Working with a designer is definitely the way ...  
3  I was blown away by the experience of working ...  
4  Having someone with extensive work experience ...  


In [14]:
# Choose a summarization model (BART is a good choice for summarization)
model_name = "facebook/bart-large-cnn"  # Or try "t5-small" or "t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [15]:
# Function to summarize a single review
def summarize_review(review_text):
    # Ensure review_text is a string (in case it's not)
    review_text = str(review_text)
    
    inputs = tokenizer.encode("summarize: " + review_text, return_tensors="pt", max_length=512, truncation=True)
    summary_ids = model.generate(inputs, max_length=150, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [16]:
# Apply summarization to the 'reviews' column and store the results
data['summary'] = data['reviews'].apply(summarize_review)

# Show a few rows of the summarized data
print(data[['url', 'reviews', 'summary']].head())

# Optionally, save the summarized data to a new CSV
output_file = "text_summarization/summarized_reviews.csv"
data.to_csv(output_file, index=False)
print(f"Summarized reviews saved to {output_file}")

                                                 url  \
0  https://www.angi.com/companylist/us/tx/argyle/...   
1                                                NaN   
2  https://www.angi.com/companylist/us/md/rosedal...   
3  https://www.angi.com/companylist/us/md/bethesd...   
4  https://www.angi.com/companylist/us/or/beavert...   

                                             reviews  \
0  Although most of the issues were not safety co...   
1  I am very pleased with the quality of work. My...   
2  Working with a designer is definitely the way ...   
3  I was blown away by the experience of working ...   
4  Having someone with extensive work experience ...   

                                             summary  
0  Most of the issues were not safety concerns an...  
1  I am very pleased with the quality of work. My...  
2  "I feel I got much more than I paid for. The k...  
3  "I was blown away by the experience of working...  
4  Having someone with extensive work experience ..